# Sistema especialista para previsão de ocupação das sessões do Instituto Noah

**Aluno:** Gabriel Lima - 38180812    
**Disciplina:** Inteligência Artificial, Linguagens Formais e Autômatos  
**Professora:** Kadidja Valéria

## Estudo inicial

1. **O que é aprendizado de máquina?** É quando um modelo aprende padrões a partir de exemplos e usa o que aprendeu para fazer previsões em novos casos. Neste trabalho, usei uma árvore de decisão para prever a faixa de ocupação final de uma sessão.

2. **O que caracteriza o aprendizado supervisionado?** O modelo treina com exemplos que já têm uma resposta conhecida. Aqui, cada sessão simulada tem dados de vendas antecipadas e uma faixa de ocupação final calculada na simulação.

3. **O que são atributos e rótulos?** Os atributos são as informações que o modelo recebe. Usei a ocupação atual da sessão, os ingressos vendidos nos últimos sete dias e os dias restantes até o espetáculo. O rótulo é `faixa_final`, que pode ser **Baixa** (abaixo de 50%), **Média** (de 50% a menos de 80%) ou **Alta** (80% ou mais).

4. **Qual é a diferença entre classificação e regressão?** A classificação prevê uma categoria, como ocupação baixa, média ou alta. A regressão prevê um valor numérico, como a taxa final de ocupação em porcentagem. Neste trabalho, usei classificação porque o modelo prevê uma faixa.

5. **Por que separar treino e teste?** O conjunto de treino serve para a árvore aprender. O conjunto de teste serve para verificar como ela se sai com exemplos que ainda não viu. Separei 75% das sessões simuladas para treino e 25% para teste.

**Previsão de ocupação das sessões do Instituto Noah**

Este trabalho foi inspirado no sistema de ingressos que desenvolvi para o Instituto Noah. Como as compras começam semanas antes do espetáculo, a equipe pode acompanhar as vendas e decidir se precisa reforçar a divulgação de alguma sessão.

A árvore de decisão usa a ocupação atual, as vendas dos últimos sete dias e os dias restantes para prever se a ocupação final será baixa, média ou alta. O sistema usa essa previsão para recomendar uma ação à equipe.

O BI da Mostra Estações registrou 59 ingressos na sessão das 18h30 e 64 na sessão das 20h30. Esses números mostram os resultados finais de uma edição. Para treinar e testar a árvore, usei sessões simuladas, pois o BI não apresenta o histórico diário de vendas de várias sessões.

**Criar os dados simulados**

In [ ]:
import random
import pandas as pd

random.seed(42)
sessoes = []

for _ in range(300):
    capacidade = 95
    dias_restantes = random.randint(1, 21)
    vendidos_agora = random.randint(8, 94)
    vendas_ultimos_7_dias = random.randint(0, min(vendidos_agora, 24))

    # Simula vendas futuras usando o ritmo recente e uma variação.
    ritmo_diario = vendas_ultimos_7_dias / 7
    variacao = random.uniform(0.5, 1.5)
    vendas_futuras = round(ritmo_diario * dias_restantes * variacao)
    vendidos_no_final = min(capacidade, vendidos_agora + vendas_futuras)

    ocupacao_final = vendidos_no_final / capacidade

    if ocupacao_final < 0.50:
        faixa_final = "Baixa"
    elif ocupacao_final < 0.80:
        faixa_final = "Média"
    else:
        faixa_final = "Alta"

    sessoes.append({
        "capacidade": capacidade,
        "vendidos_agora": vendidos_agora,
        "ocupacao_atual": vendidos_agora / capacidade,
        "vendas_ultimos_7_dias": vendas_ultimos_7_dias,
        "dias_restantes": dias_restantes,
        "vendidos_no_final": vendidos_no_final,
        "faixa_final": faixa_final
    })

dados = pd.DataFrame(sessoes)
dados.to_csv("sessoes_simuladas_noah.csv", index=False)

display(dados.head())
print(dados["faixa_final"].value_counts())

,capacidade,vendidos_agora,ocupacao_atual,vendas_ultimos_7_dias,dias_restantes,vendidos_no_final,faixa_final
0,95,22,0.231579,0,21,22,Baixa
1,95,36,0.378947,4,8,42,Baixa
2,95,19,0.200000,18,18,62,Média
3,95,19,0.200000,6,1,20,Baixa
4,95,11,0.115789,8,20,27,Baixa


faixa_final
Alta     133
Baixa     90
Média     77
Name: count, dtype: int64


**Treinar a árvore de decisão**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

entradas = ["ocupacao_atual", "vendas_ultimos_7_dias", "dias_restantes"]
classes = ["Baixa", "Média", "Alta"]

X = dados[entradas]
y = dados["faixa_final"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(X_treino, y_treino)

previsoes = modelo.predict(X_teste)

print(f"Acurácia na base simulada: {accuracy_score(y_teste, previsoes):.1%}")
matriz = confusion_matrix(y_teste, previsoes, labels=classes)

display(pd.DataFrame(
    matriz,
    index=["Real: Baixa", "Real: Média", "Real: Alta"],
    columns=["Prevista: Baixa", "Prevista: Média", "Prevista: Alta"]
))

Acurácia na base simulada: 76.0%


,Prevista: Baixa,Prevista: Média,Prevista: Alta
Real: Baixa,17,4,2
Real: Média,3,14,2
Real: Alta,0,7,26


**Aplicar as quatro regras**

In [ ]:
def analisar_sessao(vendidos, vendas_ultimos_7_dias,
                   dias_restantes, vagas_outra_sessao):

    capacidade = 95  # Capacidade adotada para a simulação.

    if dias_restantes is None:
        return "Quantos dias faltam para o espetáculo?"

    vagas = capacidade - vendidos

    exemplo = pd.DataFrame(
        [[vendidos / capacidade, vendas_ultimos_7_dias, dias_restantes]],
        columns=entradas
    )
    previsao = modelo.predict(exemplo)[0]

    if previsao == "Baixa" and dias_restantes >= 7:
        acao = "R1: reforce a divulgação da sessão e reavalie em uma semana."
    elif previsao == "Baixa":
        acao = "R2: faça uma divulgação de última hora para essa sessão."
    elif previsao == "Alta" and vagas <= 10 and vagas_outra_sessao > 0:
        acao = "R3: dê mais destaque à outra sessão, que ainda tem vagas."
    else:
        acao = "R4: mantenha a divulgação e acompanhe o ritmo das vendas."

    return f"Ocupação final prevista: {previsao}\n{acao}"

**Testar os casos da atividade**

In [ ]:
print("Caso 1 — Poucas vendas e tempo para divulgar")
print(analisar_sessao(15, 3, 14, 40))

print("\nCaso 2 — Vendas em andamento")
print(analisar_sessao(55, 8, 7, 25))

print("\nCaso 3 — Sessão perto de lotar")
print(analisar_sessao(90, 15, 7, 30))

print("\nCaso extra — Informação ausente")
print(analisar_sessao(20, 5, None, 30))

Caso 1 — Poucas vendas e tempo para divulgar
Ocupação final prevista: Baixa
R1: reforce a divulgação da sessão e reavalie em uma semana.

Caso 2 — Vendas em andamento
Ocupação final prevista: Média
R4: mantenha a divulgação e acompanhe o ritmo das vendas.

Caso 3 — Sessão perto de lotar
Ocupação final prevista: Alta
R3: dê mais destaque à outra sessão, que ainda tem vagas.

Caso extra — Informação ausente
Quantos dias faltam para o espetáculo?


## Análise dos casos

**Caso 1 - Poucas vendas.** A sessão tinha 15 ingressos vendidos e ainda faltavam 14 dias para o espetáculo. A árvore previu ocupação baixa. Como ainda dava tempo de agir, a regra R1 sugeriu reforçar a divulgação desse horário e olhar as vendas de novo depois de uma semana.

**Caso 2 - Vendas em andamento.** Nesse caso, já tinham sido vendidos 55 ingressos e faltavam sete dias. A previsão foi de ocupação média. A regra R4 orientou manter a divulgação e acompanhar o ritmo das vendas.

**Caso 3 - Sessão perto de lotar.** A sessão tinha 90 ingressos vendidos de um total de 95 lugares. A árvore previu ocupação alta e, como a outra sessão ainda tinha 30 vagas, a regra R3 sugeriu dar mais destaque ao outro horário.

**Caso extra - Informação faltando.** Deixei o número de dias em branco para ver como o programa reagiria. Ele perguntou quantos dias faltavam para o espetáculo antes de fazer a previsão.

No terceiro caso, dá para explicar o raciocínio nos dois sentidos. Partindo dos dados, a previsão alta, os cinco lugares livres e as vagas na outra sessão acionaram a R3. Partindo da recomendação de divulgar o outro horário, voltei à R3 para conferir se essas três condições estavam presentes.

Nos 75 exemplos usados no teste, a árvore acertou 57, o que deu **76% de acurácia**. O erro mais frequente foi prever ocupação média em sete casos que terminaram com ocupação alta na simulação. Como usei dados simulados, esse resultado vale para os exemplos do trabalho. Para saber como o modelo se sairia no Instituto Noah, eu precisaria testar com o histórico de vendas de várias sessões reais.

In [20]:
import ipywidgets as widgets
from IPython.display import display, clear_output

estilo = {"description_width": "180px"}
tamanho = widgets.Layout(width="550px")

vendidos = widgets.IntSlider(
    value=15,
    min=0,
    max=95,
    description="Já vendidos:",
    style=estilo,
    layout=tamanho
)

vendas_semana = widgets.IntSlider(
    value=3,
    min=0,
    max=15,
    description="Vendas nos últimos 7 dias:",
    style=estilo,
    layout=tamanho
)

dias = widgets.IntSlider(
    value=14,
    min=1,
    max=21,
    description="Dias até o espetáculo:",
    style=estilo,
    layout=tamanho
)

vagas_outra = widgets.IntSlider(
    value=40,
    min=0,
    max=95,
    description="Vagas na outra sessão:",
    style=estilo,
    layout=tamanho
)

botao = widgets.Button(
    description="Analisar sessão",
    button_style="success"
)

saida = widgets.Output()


# As vendas recentes fazem parte do total vendido.
def ajustar_limite(mudanca):
    vendas_semana.max = min(mudanca["new"], 24)


vendidos.observe(ajustar_limite, names="value")


def mostrar_resultado(clique):
    with saida:
        clear_output(wait=True)

        resultado = analisar_sessao(
            vendidos.value,
            vendas_semana.value,
            dias.value,
            vagas_outra.value
        )

        ocupacao = vendidos.value / 95 * 100

        print(f"Ingressos vendidos: {vendidos.value} de 95")
        print(f"Ocupação atual: {ocupacao:.1f}%")
        print(f"Vagas nesta sessão: {95 - vendidos.value}")
        print()
        print(resultado)
        print()
        print("Modelo treinado com dados simulados.")


botao.on_click(mostrar_resultado)

display(
    widgets.HTML("<h3>Assistente de divulgação — Instituto Noah</h3>"),
    vendidos,
    vendas_semana,
    dias,
    vagas_outra,
    botao,
    saida
)

HTML(value='<h3>Assistente de divulgação — Instituto Noah</h3>')

IntSlider(value=15, description='Já vendidos:', layout=Layout(width='550px'), max=95, style=SliderStyle(descri…

IntSlider(value=3, description='Vendas nos últimos 7 dias:', layout=Layout(width='550px'), max=15, style=Slide…

IntSlider(value=14, description='Dias até o espetáculo:', layout=Layout(width='550px'), max=21, min=1, style=S…

IntSlider(value=40, description='Vagas na outra sessão:', layout=Layout(width='550px'), max=95, style=SliderSt…

Button(button_style='success', description='Analisar sessão', style=ButtonStyle())

Output()